#CSCE 5210 FUNDAMENTALS OF AI

NAME: NAGA SAMPATH MADDINENI

STUDENT ID: 11751165

#Importing the pytorch libraries.

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import random_split, DataLoader

In [2]:
# 1. Check device (GPU if available, else CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


In [3]:
# 2. Define transforms for MNIST: convert to Tensor, normalize
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

In [4]:
# 3. Download the MNIST dataset
#    - 'train=True' fetches 60k training images, which we'll split into train/val
#    - 'train=False' fetches the 10k test images
full_train_dataset = torchvision.datasets.MNIST(
    root='./data', train=True, transform=transform, download=True)
test_dataset = torchvision.datasets.MNIST(
    root='./data', train=False, transform=transform, download=True)

# Training, Testing, Validation set splitting:

In [5]:
# 4. Split into training (50k) and validation (10k) sets
train_dataset, val_dataset = random_split(full_train_dataset, [50000, 10000])

In [6]:
# 5. Create Data Loaders (batch size = 64)
batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# Defining the hidden layers:

In [7]:
# 6. Define the DeepMLP class with 3 hidden layers and dropout=0.3 after first hidden layer
class DeepMLP(nn.Module):
    def __init__(self):
        super(DeepMLP, self).__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(28*28, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 64)
        self.fc4 = nn.Linear(64, 10)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(p=0.3)

    def forward(self, x):
        x = self.flatten(x)
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.relu(self.fc2(x))
        x = self.relu(self.fc3(x))
        x = self.fc4(x)
        return x

# Instantiate model, move to device
model = DeepMLP().to(device)
print(model)

DeepMLP(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (fc1): Linear(in_features=784, out_features=256, bias=True)
  (fc2): Linear(in_features=256, out_features=128, bias=True)
  (fc3): Linear(in_features=128, out_features=64, bias=True)
  (fc4): Linear(in_features=64, out_features=10, bias=True)
  (relu): ReLU()
  (dropout): Dropout(p=0.3, inplace=False)
)


In [8]:
# 7. Define loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Defining the Evaluation of the Model:

In [9]:
# 8. Helper function for validation/test evaluation
def evaluate(model, loader):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item()*images.size(0)
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)
    return total_loss / total, correct / total


# Training the model:

In [10]:
# 9. Main training loop for 5 epochs
num_epochs = 5
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()  # Reset gradients
        outputs = model(images)  # Forward pass
        loss = criterion(outputs, labels)  # Compute loss
        loss.backward()  # Backward pass
        optimizer.step()  # Update parameters

        running_loss += loss.item()*images.size(0)

    avg_train_loss = running_loss / len(train_loader.dataset)
    val_loss, val_accuracy = evaluate(model, val_loader)

    print(f"Epoch [{epoch+1}/{num_epochs}], Train Loss: {avg_train_loss:.4f}, Val Loss: {val_loss:.4f}, Val Acc: {val_accuracy*100:.2f}%")

Epoch [1/5], Train Loss: 0.3256, Val Loss: 0.1454, Val Acc: 95.36%
Epoch [2/5], Train Loss: 0.1497, Val Loss: 0.1108, Val Acc: 96.53%
Epoch [3/5], Train Loss: 0.1162, Val Loss: 0.0897, Val Acc: 97.06%
Epoch [4/5], Train Loss: 0.0985, Val Loss: 0.0798, Val Acc: 97.52%
Epoch [5/5], Train Loss: 0.0861, Val Loss: 0.0743, Val Acc: 97.60%


# Evaluating the Testdata to get the accuracy:

In [11]:
# 10. Final test evaluation
test_loss, test_accuracy = evaluate(model, test_loader)
print(f"Test Loss: {test_loss:.4f}, Test Accuracy: {test_accuracy*100:.2f}%")

Test Loss: 0.0739, Test Accuracy: 97.80%
